# Netflix Shows Data ETL on GCP
This notebook is working on a ETL pipeline design for netflix shows and movies data from Kaggle, enriching the data from the open source API in OMDb.

*Source*: [Netflix TV Shows and Movies](https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies)

### 1. Import Necessary Libraries

In [1]:
import requests
import json
import os
import pandas as pd

import time

### 2. Read the CSV File and Import the Data as a Pandas Dataframe

In [2]:
df_titles = pd.read_csv('/home/jasonzelin/data-analytics-portfolio/[in_progress]_netflix_data_etl_on_gcp/data/titles.csv')
df_credits = pd.read_csv('/home/jasonzelin/data-analytics-portfolio/[in_progress]_netflix_data_etl_on_gcp/data/credits.csv')

In [3]:
print('Titles:\n') 
print(df_titles.info(), '\n')
print('Credits:\n') 
print(df_credits.info())

Titles:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   object 
 1   title                 5849 non-null   object 
 2   type                  5850 non-null   object 
 3   description           5832 non-null   object 
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   object 
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   object 
 8   production_countries  5850 non-null   object 
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   object 
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  tmdb_score            5539 non-null   float64
dtypes: float64(5

### 3. Enriching the data from Kaggle Dataset with OMDb API Data

In [4]:
my_api_key = 'fd7f3858'
base_url = 'http://www.omdbapi.com/'
title = ''

params = {
    'apikey': my_api_key,
    't': title
}

In [ ]:
titles_list = missing_titles
omdb_data = []

for i in titles_list:
    params['t'] = i
    response = requests.get(base_url, params=params)
    data = response.json()
    if data.get("Error") == "Request limit reached!":
        print("⚠️ API daily limit reached. Breaking loop.")
        break
    omdb_data.append(data)
    time.sleep(0.2)  # To respect API rate limits

for i, v in enumerate(omdb_data):
    omdb_data[i]['original_title'] = titles_list[i]

⚠️ API daily limit reached. Breaking loop.


In [44]:
omdb_data

[{'Response': 'False',
  'Error': 'Movie not found!',
  'original_title': 'Five Came Back: The Reference Films'},
 {'Title': "Monty Python's Life of Brian",
  'Year': '1979',
  'Rated': 'R',
  'Released': '17 Aug 1979',
  'Runtime': '94 min',
  'Genre': 'Comedy',
  'Director': 'Terry Jones',
  'Writer': 'Graham Chapman, John Cleese, Terry Gilliam',
  'Actors': 'Graham Chapman, John Cleese, Michael Palin',
  'Plot': 'Born on the original Christmas in the stable next door to Jesus Christ, Brian of Nazareth spends his life being mistaken for a messiah.',
  'Language': 'English, Latin',
  'Country': 'United Kingdom',
  'Awards': 'N/A',
  'Poster': 'https://m.media-amazon.com/images/M/MV5BNDMzY2E4NjEtMTJiZC00Y2UzLWFiM2MtZWVhNTg5OGQxNjk1XkEyXkFqcGc@._V1_SX300.jpg',
  'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.0/10'},
   {'Source': 'Rotten Tomatoes', 'Value': '96%'},
   {'Source': 'Metacritic', 'Value': '77/100'}],
  'Metascore': '77',
  'imdbRating': '8.0',
  'imdbVotes': 

In [46]:
def remove_request_limit_reached(data):
    try:
        if data['Error'] == 'Request limit reached!':
            return False
        elif data['Error'] == 'Movie not found!':
            return True
    except KeyError:
        return True


omdb_data_cleaned = [i for i in omdb_data if remove_request_limit_reached(i)]
len(omdb_data_cleaned)

1001

In [47]:
omdb_data_cleaned

[{'Response': 'False',
  'Error': 'Movie not found!',
  'original_title': 'Five Came Back: The Reference Films'},
 {'Title': "Monty Python's Life of Brian",
  'Year': '1979',
  'Rated': 'R',
  'Released': '17 Aug 1979',
  'Runtime': '94 min',
  'Genre': 'Comedy',
  'Director': 'Terry Jones',
  'Writer': 'Graham Chapman, John Cleese, Terry Gilliam',
  'Actors': 'Graham Chapman, John Cleese, Michael Palin',
  'Plot': 'Born on the original Christmas in the stable next door to Jesus Christ, Brian of Nazareth spends his life being mistaken for a messiah.',
  'Language': 'English, Latin',
  'Country': 'United Kingdom',
  'Awards': 'N/A',
  'Poster': 'https://m.media-amazon.com/images/M/MV5BNDMzY2E4NjEtMTJiZC00Y2UzLWFiM2MtZWVhNTg5OGQxNjk1XkEyXkFqcGc@._V1_SX300.jpg',
  'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.0/10'},
   {'Source': 'Rotten Tomatoes', 'Value': '96%'},
   {'Source': 'Metacritic', 'Value': '77/100'}],
  'Metascore': '77',
  'imdbRating': '8.0',
  'imdbVotes': 

In [48]:
omdb_data_cleaned_df = pd.read_json(json.dumps(omdb_data_cleaned))
omdb_data_cleaned_df.to_csv('/home/jasonzelin/data-analytics-portfolio/[in_progress]_netflix_data_etl_on_gcp/data/omdb_data_batch1.csv', index=False)
omdb_data_cleaned_df

/tmp/ipykernel_12787/1116496920.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  omdb_data_cleaned_df = pd.read_json(json.dumps(omdb_data_cleaned))


,Response,Error,original_title,Title,Year,Rated,Released,Runtime,Genre,Director,...,Metascore,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,totalSeasons
0,False,Movie not found!,Five Came Back: The Reference Films,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,NaN,Life of Brian,Monty Python's Life of Brian,1979,R,17 Aug 1979,94 min,Comedy,Terry Jones,...,77,8.0,"430,381",tt0079470,movie,N/A,"$20,206,622",N/A,N/A,NaN
2,False,Movie not found!,Alexandria… Why?,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,Movie not found!,Saladin the Victorious,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,Movie not found!,"Beirut, Oh Beirut",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,True,NaN,The Forest,The Forest,2016,PG-13,08 Jan 2016,93 min,"Horror, Mystery, Thriller",Jason Zada,...,34,4.8,"46,521",tt3387542,movie,N/A,"$26,594,261",N/A,N/A,NaN
997,True,NaN,Cable Girls,Cable Girls,2017–2020,TV-MA,28 Apr 2017,1 min,"Drama, History",N/A,...,N/A,7.5,"17,173",tt5674718,series,NaN,NaN,NaN,NaN,5
998,False,Movie not found!,Jim & Andy: The Great Beyond - Featuring a Ver...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
999,True,NaN,A-X-L,A-X-L,2018,PG,24 Aug 2018,98 min,"Action, Adventure, Drama",Oliver Daly,...,29,5.3,"14,179",tt5709188,movie,N/A,"$6,501,381",N/A,N/A,NaN


In [66]:
omdb_data_cleaned_csv = pd.DataFrame()
parent_dir = '/home/jasonzelin/data-analytics-portfolio/[in_progress]_netflix_data_etl_on_gcp/data/'
files = os.listdir(parent_dir)
for file in files:
    if file.startswith('omdb_data_batch') and file.endswith('.csv'):
        temp_df = pd.read_csv(f'{parent_dir}{file}')
        omdb_data_cleaned_csv = pd.concat([omdb_data_cleaned_csv, temp_df], ignore_index=True)

def fill_title(row):
    if (pd.isnull(row['omdb_title'])) & (row['Error'] == 'Movie not found!'):
        return row['original_title']
    else:
        return row['omdb_title']

comparison_df = df_titles.merge(right=omdb_data_cleaned_csv, how='left', left_on='title', right_on='original_title').rename(columns={'Title': 'omdb_title'})
comparison_df = comparison_df.assign(omdb_title = lambda x: x.apply(fill_title, axis=1))
comparison_df

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,...,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,totalSeasons,Unnamed: 0
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,...,8.2,"989,092",tt0075314,movie,NaN,"$28,262,574",NaN,NaN,NaN,0.0
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,...,7.6,"125,930",tt0068473,movie,NaN,NaN,NaN,NaN,NaN,1.0
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,...,8.2,"593,325",tt0071853,movie,NaN,"$2,562,392",NaN,NaN,NaN,2.0
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,...,7.7,"82,152",tt0061578,movie,NaN,NaN,NaN,NaN,NaN,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5859,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021,NaN,100,"['romance', 'drama']",['NG'],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5860,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021,NaN,134,['drama'],[],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5861,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,NaN,90,['comedy'],['CO'],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5862,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,[],['US'],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
print(f'{comparison_df[comparison_df['omdb_title'].isnull()].shape[0]} titles out of {comparison_df.shape[0]} from titles.csv still missing from OMDB API data.')

3946 titles out of 5864 from titles.csv still missing from OMDB API data.


In [68]:
missing_titles = comparison_df[comparison_df['omdb_title'].isnull()]['title'].tolist()
print(missing_titles)

['Aggretsuko', 'Malevolent', 'Jack Whitehall: Travels with My Father', 'Two Sentence Horror Stories', 'Dirty Money', 'Requiem', 'The Rain', "Sunderland 'Til I Die", 'Happy as Lazzaro', 'Nailed It!', 'My Mister', 'Bakugan: Battle Planet', 'Everybody Knows', 'Dave Chappelle', 'Damnation', 'Innocent', 'Great News', '22 July', 'Falls Around Her', 'Better Than Us', 'Insatiable', 'Sand Castle', 'Zoids Wild', 'Explained', "Buster's Mal Heart", 'The Toys That Made Us', 'On Body and Soul', 'Ben Is Back', 'Perfume', 'Mr. Sunshine', 'Private Life', 'Mirage', 'Ravenous', 'RBG', 'An Evening with Beverly Luff Linn', 'Friends from College', 'Mary Magdalene', 'Disjointed', 'Sisters', "Dumplin'", 'Tau', 'Creeped Out', 'Skins', 'Wormwood', 'Duck Butter', 'Never Stop Dreaming: The Life and Legacy of Shimon Peres', "The Zookeeper's Wife", 'Game Over, Man!', 'Polly Pocket', 'Steve Martin and Martin Short: An Evening You Will Forget for the Rest of Your Life', 'Victoria & Abdul', 'Hampstead', 'Hannah Gadsby